In [3]:
import json
import re
from pprint import pprint

import json
import re
from pprint import pprint

# --- Configuration ---
system_a = "lightrag"
system_b = "muvera"
cls = "agri3"

# --- Load Data ---
# Evaluation 1: The judge sees [system_a, system_b]
try:
    with open(f'results/eval/{cls}_evaluation_{system_a}_{system_b}.json', 'r', encoding='utf-8') as file:
        data_a_b = json.load(file)
except FileNotFoundError:
    print(f"Error: File not found for {system_a} vs {system_b}. Please check the path.")
    exit()

# Evaluation 2: The judge sees [system_b, system_a]
try:
    with open(f'results/eval/{cls}_evaluation_{system_b}_{system_a}.json', 'r', encoding='utf-8') as file:
        data_b_a = json.load(file)
except FileNotFoundError:
    print(f"Error: File not found for {system_b} vs {system_a}. Please check the path.")
    exit()

# --- Validation ---
if len(data_a_b) != len(data_b_a):
    print("Warning: The two evaluation files have a different number of records.")
    print(f"File 1 ({system_a}_{system_b}) has {len(data_a_b)} records.")
    print(f"File 2 ({system_b}_{system_a}) has {len(data_b_a)} records.")
    # Consider exiting or truncating to the smaller size
    # For now, we'll proceed with the smaller of the two
    min_len = min(len(data_a_b), len(data_b_a))
    data_a_b = data_a_b[:min_len]
    data_b_a = data_b_a[:min_len]

print(f"Processing {len(data_a_b)} pairs of evaluation records.")

# --- Initialize Results Dictionary ---
# This structure clearly tracks strong wins and ties.
results_dict = {
    "Comprehensiveness": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Diversity": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Empowerment": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Overall Winner": {system_a: 0, system_b: 0, "Tie/Bias": 0},
}
categories = list(results_dict.keys())

def parse_evaluation(evaluation_str: str) -> dict:
    """Extracts the JSON object from the LLM's string output."""
    match = re.search(r'\{.*\}', evaluation_str, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return None # Return None on parsing error
    return None

# --- Main Logic ---
for i, (eval1, eval2) in enumerate(zip(data_a_b, data_b_a)):
    # Parse the evaluation results for both runs
    parsed_eval1 = parse_evaluation(eval1["evaluation"]) # Order: [A, B]
    parsed_eval2 = parse_evaluation(eval2["evaluation"]) # Order: [B, A]
    
    if not parsed_eval1 or not parsed_eval2:
        print(f"Skipping record {i} due to a JSON parsing error in one or both evaluations.")
        continue

    # Apply counterbalancing logic for each category
    for category in categories:
        if category not in parsed_eval1 or category not in parsed_eval2:
            print(f"Skipping category '{category}' for record {i} as it's missing from an evaluation.")
            continue
            
        winner1 = parsed_eval1[category].get("Winner") # Winner when order is [A, B]
        winner2 = parsed_eval2[category].get("Winner") # Winner when order is [B, A]

        # Case 1: Strong Win for system_a (lightrag)
        # It won as "Answer 1" in the first eval and as "Answer 2" in the second.
        if winner1 == "Answer 1" and winner2 == "Answer 2":
            results_dict[category][system_a] += 1
        
        # Case 2: Strong Win for system_b (colbert)
        # It won as "Answer 2" in the first eval and as "Answer 1" in the second.
        elif winner1 == "Answer 2" and winner2 == "Answer 1":
            results_dict[category][system_b] += 1
        
        # Case 3: All other outcomes are considered a Tie or reveal positional bias
        # This includes (A, A), (B, B), or if one evaluation was a "Tie".
        else:
            results_dict[category]["Tie/Bias"] += 1

# --- Display Results ---
print("\n--- Counterbalanced Evaluation Results ---")
pprint(results_dict)

for k,v in results_dict.items():
    print(f"Category:{k}")
    total = v[system_a] + v[system_b]
    print(f"{system_a}: {v[system_a]/total}, {system_b}: {v[system_b]/total}")

Processing 500 pairs of evaluation records.
Skipping record 24 due to a JSON parsing error in one or both evaluations.
Skipping record 53 due to a JSON parsing error in one or both evaluations.
Skipping record 64 due to a JSON parsing error in one or both evaluations.
Skipping record 92 due to a JSON parsing error in one or both evaluations.
Skipping record 222 due to a JSON parsing error in one or both evaluations.
Skipping record 266 due to a JSON parsing error in one or both evaluations.
Skipping record 270 due to a JSON parsing error in one or both evaluations.
Skipping record 340 due to a JSON parsing error in one or both evaluations.
Skipping record 418 due to a JSON parsing error in one or both evaluations.
Skipping record 492 due to a JSON parsing error in one or both evaluations.

--- Counterbalanced Evaluation Results ---
{'Comprehensiveness': {'Tie/Bias': 208, 'lightrag': 154, 'muvera': 128},
 'Diversity': {'Tie/Bias': 187, 'lightrag': 120, 'muvera': 183},
 'Empowerment': {'

In [ ]:
# --- Configuration ---
system_a = "colbert_query"
system_b = "lightrag"
cls = "agri3"

# --- Load Data ---
# Evaluation 1: The judge sees [system_a, system_b]
try:
    with open(f'results/eval/{cls}_evaluation_{system_a}_{system_b}.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
except FileNotFoundError:
    print(f"Error: File not found for {system_a} vs {system_b}. Please check the path.")
    exit()

print(f"Processing {len(data_a_b)} pairs of evaluation records.")

# --- Initialize Results Dictionary ---
# This structure clearly tracks strong wins and ties.
results_dict = {
    "Comprehensiveness": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Diversity": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Empowerment": {system_a: 0, system_b: 0, "Tie/Bias": 0},
    "Overall Winner": {system_a: 0, system_b: 0, "Tie/Bias": 0},
}
categories = list(results_dict.keys())

def parse_evaluation(evaluation_str: str) -> dict:
    """Extracts the JSON object from the LLM's string output."""
    match = re.search(r'\{.*\}', evaluation_str, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return None # Return None on parsing error
    return None

# --- Main Logic ---
for i, eval1 in enumerate(data):
    # Parse the evaluation results for both runs
    parsed_eval1 = parse_evaluation(eval1["evaluation"]) # Order: [A, B]
    
    if not parsed_eval1:
        print(f"Skipping record {i} due to a JSON parsing error in one or both evaluations.")
        continue

    # Apply counterbalancing logic for each category
    for category in categories:
        if category not in parsed_eval1 or category not in parsed_eval2:
            print(f"Skipping category '{category}' for record {i} as it's missing from an evaluation.")
            continue
            
        winner1 = parsed_eval1[category].get("Winner") # Winner when order is [A, B]
        winner2 = parsed_eval2[category].get("Winner") # Winner when order is [B, A]

        # Case 1: Strong Win for system_a (lightrag)
        # It won as "Answer 1" in the first eval and as "Answer 2" in the second.
        if winner1 == "Answer 1":
            results_dict[category][system_a] += 1
        
        # Case 2: Strong Win for system_b (colbert)
        # It won as "Answer 2" in the first eval and as "Answer 1" in the second.
        elif winner1 == "Answer 2":
            results_dict[category][system_b] += 1
        
        # Case 3: All other outcomes are considered a Tie or reveal positional bias
        # This includes (A, A), (B, B), or if one evaluation was a "Tie".
        else:
            results_dict[category]["Tie/Bias"] += 1

# --- Display Results ---
print("\n--- Counterbalanced Evaluation Results ---")
pprint(results_dict)

for k,v in results_dict.items():
    print(f"Category:{k}")
    total = v[system_a] + v[system_b]
    print(f"{system_a}: {v[system_a]/total}, {system_b}: {v[system_b]/total}")


Processing 500 pairs of evaluation records.

--- Counterbalanced Evaluation Results ---
{'Comprehensiveness': {'Tie/Bias': 41, 'colbert_query': 226, 'lightrag': 233},
 'Diversity': {'Tie/Bias': 21, 'colbert_query': 189, 'lightrag': 290},
 'Empowerment': {'Tie/Bias': 8, 'colbert_query': 200, 'lightrag': 292},
 'Overall Winner': {'Tie/Bias': 0, 'colbert_query': 219, 'lightrag': 281}}
Category:Comprehensiveness
colbert_query: 0.4923747276688453, lightrag: 0.5076252723311547
Category:Diversity
colbert_query: 0.3945720250521921, lightrag: 0.605427974947808
Category:Empowerment
colbert_query: 0.4065040650406504, lightrag: 0.5934959349593496
Category:Overall Winner
colbert_query: 0.438, lightrag: 0.562


In [10]:
with open("contexts/colbert_agri3_query_contexts.json", 'r', encoding='utf-8') as file:
        data = json.load(file)
data[0]["result"]

'-----Entities(KG)-----\n\n```json\n[{"id": "1", "entity": "Experienced Beekeeper", "type": "person", "description": "An experienced beekeeper is a knowledgeable individual who has practiced the hobby for several years and can serve as a valuable resource, particularly for novices. They can be consulted to observe a beekeeper\'s practices and techniques to help identify the causes of persistent problems, such as issues with a bee colony\'s temperament. An experienced beekeeper can also act as a potential mentor, though it is cautioned that years of experience do not always guarantee competence or teaching ability, so a mentor should be chosen with care. Furthermore, unlike new beekeepers, they may have the opportunity to turn the hobby into a sideline business, such as selling honey or providing crop pollination services.", "created_at": "2025-07-29 08:05:16", "file_path": "unknown_source"}, {"id": "2", "entity": "Neighbors", "type": "category", "description": "Based on the provided de

In [ ]:
import json
import re

def escape_latex(text):
    """Escape special LaTeX characters in text"""
    if not text:
        return ""
    
    # Replace special LaTeX characters
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '^': r'\textasciicircum{}',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
    }
    
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    
    return text

def extract_entities_to_latex(json_data):
    """Extract entities from JSON and format as LaTeX table"""
    entities = json_data.get('entities', [])
    
    latex_content = []
    latex_content.append(r"\begin{longtable}{|p{3cm}|p{2cm}|p{9cm}|}")
    latex_content.append(r"\hline")
    latex_content.append(r"\textbf{Entity} & \textbf{Type} & \textbf{Description} \\")
    latex_content.append(r"\hline")
    
    for entity in entities:
        name = escape_latex(entity.get('entity', ''))
        entity_type = escape_latex(entity.get('type', ''))
        description = escape_latex(entity.get('description', ''))
        
        # Truncate very long descriptions
        if len(description) > 500:
            description = description[:500] + "..."
        
        latex_content.append(f"{name} & {entity_type} & {description} \\\\")
        latex_content.append(r"\hline")
    
    latex_content.append(r"\end{longtable}")
    
    return "\n".join(latex_content)

def extract_relationships_to_latex(json_data):
    """Extract relationships from JSON and format as LaTeX table"""
    relationships = json_data.get('relationships', [])
    
    latex_content = []
    latex_content.append(r"\begin{longtable}{|p{2.5cm}|p{2.5cm}|p{9cm}|}")
    latex_content.append(r"\hline")
    latex_content.append(r"\textbf{Entity 1} & \textbf{Entity 2} & \textbf{Relationship Description} \\")
    latex_content.append(r"\hline")
    
    for rel in relationships:
        entity1 = escape_latex(rel.get('entity1', ''))
        entity2 = escape_latex(rel.get('entity2', ''))
        description = escape_latex(rel.get('description', ''))
        
        # Truncate very long descriptions
        if len(description) > 400:
            description = description[:400] + "..."
        
        latex_content.append(f"{entity1} & {entity2} & {description} \\\\")
        latex_content.append(r"\hline")
    
    latex_content.append(r"\end{longtable}")
    
    return "\n".join(latex_content)

def extract_document_chunks_to_latex(json_data):
    """Extract document chunks from JSON and format as LaTeX sections"""
    chunks = json_data.get('document_chunks', [])
    
    latex_content = []
    
    for i, chunk in enumerate(chunks, 1):
        content = escape_latex(chunk.get('content', ''))
        
        # Create a subsection for each chunk
        latex_content.append(f"\\subsection{{Document Chunk {i}}}")
        latex_content.append(content)
        latex_content.append("")  # Empty line for spacing
    
    return "\n".join(latex_content)

def create_full_latex_document(json_data, query=""):
    """Create a complete LaTeX document from JSON data"""
    
    latex_template = r"""\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage{longtable}
\usepackage{booktabs}
\usepackage{geometry}
\usepackage{array}
\usepackage{parskip}
\geometry{margin=0.8in}

\title{Knowledge Graph: Extracted Data Analysis}
\date{\today}

\begin{document}

\maketitle

"""
    
    if query:
        latex_template += f"\\section{{Query}}\n{escape_latex(query)}\n\n"
    
    # Add entities section
    latex_template += "\\subsection{All Entities}\n\n"
    latex_template += extract_entities_to_latex(json_data)
    
    # Add relationships section
    latex_template += "\n\n\\newpage\n\\subsection{All Relationships}\n\n"
    latex_template += extract_relationships_to_latex(json_data)
    
    # Add document chunks section
    latex_template += "\n\n\\newpage\n\\subsection{Document Chunks}\n\n"
    latex_template += extract_document_chunks_to_latex(json_data)
    
    latex_template += "\n\\end{document}"
    
    return latex_template

def parse_your_data_format(text_data):
    """Parse the specific format from your document"""
    
    # Extract entities section
    entities_match = re.search(r'-----Entities\(KG\)-----\n\n```json\n(.*?)\n```', text_data, re.DOTALL)
    entities = []
    if entities_match:
        try:
            entities = json.loads(entities_match.group(1))
        except json.JSONDecodeError:
            print("Error parsing entities JSON")
    
    # Extract relationships section
    relationships_match = re.search(r'-----Relationships\(KG\)-----\n\n```json\n(.*?)\n```', text_data, re.DOTALL)
    relationships = []
    if relationships_match:
        try:
            relationships = json.loads(relationships_match.group(1))
        except json.JSONDecodeError:
            print("Error parsing relationships JSON")
    
    # Extract document chunks section
    chunks_match = re.search(r'-----Document Chunks\(DC\)-----\n\n```json\n(.*?)\n```', text_data, re.DOTALL)
    document_chunks = []
    if chunks_match:
        try:
            document_chunks = json.loads(chunks_match.group(1))
        except json.JSONDecodeError:
            print("Error parsing document chunks JSON")
    
    # Extract query
    query_match = re.search(r'"query": "(.*?)"', text_data)
    query = query_match.group(1) if query_match else ""
    
    return {
        'entities': entities,
        'relationships': relationships,
        'document_chunks': document_chunks,
        'query': query
    }


parsed_data = parse_your_data_format(data[0]["result"])

# Generate LaTeX
latex_output = create_full_latex_document(parsed_data, parsed_data['query'])

# Save to file
with open('output.tex', 'w', encoding='utf-8') as file:
    file.write(latex_output)

print("LaTeX document generated successfully!")
print(f"Found {len(parsed_data['entities'])} entities")
print(f"Found {len(parsed_data['relationships'])} relationships")
print(f"Found {len(parsed_data['document_chunks'])} document chunks")


LaTeX document generated successfully!
Found 50 entities
Found 108 relationships
Found 9 document chunks
